# Reference solution: differential cryptanalysis of 4-round DES

This notebook recovers the secret key of the 4-round DES variant (standard
DES S-boxes, no initial/final permutation) using a chosen-plaintext
differential attack, and verifies the recovered key.

Outline of the attack. Writing the cipher as
`L_i = R_{i-1}`, `R_i = L_{i-1} XOR F(R_{i-1}, K_i)`, the ciphertext is
`C = (R4, L4)`, so the input to the round-4 F function, `R3 = L4`, is read
directly from the ciphertext. From `R4 = R2 XOR F(R3, K4)` we get
`F(R3,K4) XOR F(R3*,K4) = R4' XOR R2'`. We choose plaintext pairs with right
half difference 0 (round 1 has zero F-difference) and an input difference
that activates a single S-box in round 2, so `R2'` is a known constant for
right pairs. That gives the round-4 F output difference, and a standard
S-box counting step recovers `K4`; the full key follows by inverting the key
schedule and brute-forcing the 8 bits that PC-2 discards.

## Setup: cipher tables and bit helpers

In [1]:
import collections
import os
import random
import sys

# The notebook lives in solution/; make the public DES tables/helpers in
# oracle_app importable whether we run from solution/ or the repo root.
for _cand in ('.', '..', os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(_cand, 'oracle_app')):
        sys.path.insert(0, os.path.abspath(_cand))
        break

# These are all PUBLIC DES constants/helpers (real S-boxes and tables) plus the
# encryption routine used only to VERIFY a recovered key. The secret key itself
# is never imported - it is recovered purely from oracle queries below.
from oracle_app.custom_DES import (
    S_BOXES, E, P, PC1, PC2, SHIFT_SCHEDULE,
    des_encrypt_hex, permute, left_shift, to_bin,
)

random.seed(1234)

def bits_of(val, n):
    return [(val >> (n - 1 - i)) & 1 for i in range(n)]

def from_bits(bits):
    v = 0
    for b in bits:
        v = (v << 1) | b
    return v

def expand(R):                       # 32-bit -> 48-bit
    rb = bits_of(R, 32)
    return from_bits([rb[E[k] - 1] for k in range(48)])

def permP(x):                        # 32-bit -> 32-bit
    xb = bits_of(x, 32)
    return from_bits([xb[P[k] - 1] for k in range(32)])

Pinv = [0] * 32
for _i in range(32):
    Pinv[P[_i] - 1] = _i + 1

def permPinv(y):                     # 32-bit -> 32-bit
    yb = bits_of(y, 32)
    return from_bits([yb[Pinv[k] - 1] for k in range(32)])

def sbox(idx, six):                  # 6-bit -> 4-bit
    b = bits_of(six, 6)
    row = (b[0] << 1) | b[5]
    col = (b[1] << 3) | (b[2] << 2) | (b[3] << 1) | b[4]
    return S_BOXES[idx][row][col]

for _ in range(1000):                # sanity check P / P^-1
    _x = random.getrandbits(32)
    assert permPinv(permP(_x)) == _x
print('cipher helpers ready')

cipher helpers ready


## Step 1: S-box DDT and the differential characteristic

We pick the best single-S-box differential whose 6-bit input difference has
its two outer bits zero (so the E-expansion activates exactly one S-box in
round 2 without disturbing its neighbours). We then craft the left-half
input difference `L0'` that realises it, and compute the resulting constant
`R2'` for right pairs.

In [2]:
def best_inner_differential():
    best = None  # (count, box, din, dout)
    for box in range(8):
        ddt = collections.defaultdict(lambda: collections.defaultdict(int))
        for x in range(64):
            for din in range(64):
                if din & 0b100001:        # outer bits must be zero
                    continue
                dout = sbox(box, x) ^ sbox(box, x ^ din)
                ddt[din][dout] += 1
        for din, row in ddt.items():
            if din == 0:
                continue
            for dout, c in row.items():
                if best is None or c > best[0]:
                    best = (c, box, din, dout)
    return best

CNT, BOX, DIN, DOUT = best_inner_differential()
print(f'Chosen differential: S{BOX+1}  in=0x{DIN:02x} -> out=0x{DOUT:x}  count={CNT}/64  p={CNT/64:.3f}')

def make_L0prime(box, din):
    dbits = bits_of(din, 6)              # b1..b6, with b1=b6=0
    Lp = [0] * 32
    for j in range(1, 5):               # inner bits b2..b5
        if dbits[j]:
            Lp[E[6 * box + j] - 1] = 1   # set the unique source R bit
    return from_bits(Lp)

L0p = make_L0prime(BOX, DIN)
edb = bits_of(expand(L0p), 48)          # verify only S-box BOX is active
for m in range(8):
    chunk = from_bits(edb[6 * m:6 * m + 6])
    assert chunk == (DIN if m == BOX else 0)

R2P_CONST = permP(DOUT << (4 * (7 - BOX)))   # predicted R2' for right pairs
print(f'L0\' = 0x{L0p:08X}   (right-half difference is 0)')
print(f'Predicted R2\' (right pairs) = 0x{R2P_CONST:08X}')

Chosen differential: S2  in=0x08 -> out=0xa  count=16/64  p=0.250
L0' = 0x04000000   (right-half difference is 0)
Predicted R2' (right pairs) = 0x40080000


## Step 2: the encryption oracle

This queries the **live deployed oracle** over HTTP. The attack never sees the
key; it only sends chosen plaintexts and reads ciphertexts. Set `BASE_URL` (or
the `ORACLE_URL` environment variable) to the oracle address, e.g.
`http://<SERVER_IP>` (defaults to `http://localhost` for a local deployment).

In [3]:
import requests

# Address of the deployed oracle. Override with the ORACLE_URL env var, e.g.
#   ORACLE_URL=http://192.168.1.50  jupyter nbconvert --execute ...
BASE_URL = os.environ.get('ORACLE_URL', 'http://des.rathanappana.com').rstrip('/')

_query_count = 0
_session = requests.Session()

def oracle(pt64):
    """Encrypt a 64-bit int plaintext via the live HTTP oracle; return 64-bit int."""
    global _query_count
    _query_count += 1
    resp = _session.post(
        f'{BASE_URL}/api/encrypt',
        json={'plaintext': f'{pt64:016X}'},
        timeout=30,
    )
    if resp.status_code == 429:
        raise RuntimeError('Oracle rate limit reached (1000/day per IP). Try again later.')
    resp.raise_for_status()
    return int(resp.json()['ciphertext'], 16)

# connectivity check (also confirms the API contract)
_probe = oracle(0x0123456789ABCDEF)
print(f'Connected to oracle at {BASE_URL}; sample CT = {_probe:016X}')

Connected to oracle at http://des.rathanappana.com; sample CT = F50804444FF5DE47


## Step 3: collect pairs and recover the last-round subkey K4

Each plaintext pair differs by `L0'` in the left half and `0` in the right
half. From the ciphertexts we read `R3 = L4` and `R4`, form the round-4 F
input difference `E(L4')` and the predicted output difference
`R4' XOR R2'`, and count, per S-box, the 6-bit key candidates consistent
with the differential. For every S-box except the one tied to the
characteristic the predicted output difference is correct for *all* pairs,
so those 42 key bits are essentially deterministic; the remaining S-box is
resolved by the right pairs.

In [4]:
NUM_PAIRS = 400                       # 800 queries; within the 1000/day budget
counts = [collections.Counter() for _ in range(8)]
active_pairs = [0] * 8

for _ in range(NUM_PAIRS):
    pt = random.getrandbits(64)
    pts = pt ^ (L0p << 32)            # flip L0' in the left half; R0' = 0
    C, Cs = oracle(pt), oracle(pts)

    R4, L4 = C >> 32, C & 0xFFFFFFFF
    R4s, L4s = Cs >> 32, Cs & 0xFFFFFFFF
    R3, R3s = L4, L4s                 # C = (R4, L4) so R3 = L4

    sout_diff = permPinv(R4 ^ R4s ^ R2P_CONST)   # expected round-4 S-box out diffs
    idb = bits_of(expand(R3 ^ R3s), 48)          # round-4 F input difference
    in_a = bits_of(expand(R3), 48)
    in_b = bits_of(expand(R3s), 48)
    sob = bits_of(sout_diff, 32)

    for m in range(8):
        din_m = from_bits(idb[6 * m:6 * m + 6])
        if din_m == 0:
            continue
        active_pairs[m] += 1
        dout_m = from_bits(sob[4 * m:4 * m + 4])
        a = from_bits(in_a[6 * m:6 * m + 6])
        b = from_bits(in_b[6 * m:6 * m + 6])
        for k in range(64):
            if sbox(m, a ^ k) ^ sbox(m, b ^ k) == dout_m:
                counts[m][k] += 1

rec_chunks = [counts[m].most_common(1)[0][0] for m in range(8)]
K4_rec = 0
for m in range(8):
    K4_rec = (K4_rec << 6) | rec_chunks[m]

# The most-voted candidate per S-box is the recovered K4 chunk. A large gap to
# the runner-up indicates a confident recovery (no secret is consulted here).
for m in range(8):
    top = counts[m].most_common(2)
    best_k, best_c = top[0]
    runner = top[1][1] if len(top) > 1 else 0
    print(f'S{m+1}: K4=0x{best_k:02x}  votes={best_c}/{active_pairs[m]}  (runner-up {runner})')
print(f'\nQueries used: {_query_count}')
print(f'Recovered last-round subkey K4 = {K4_rec:012X}')

S1: K4=0x22  votes=333/333  (runner-up 132)
S2: K4=0x16  votes=62/342  (runner-up 51)
S3: K4=0x39  votes=369/369  (runner-up 97)
S4: K4=0x21  votes=378/378  (runner-up 101)
S5: K4=0x26  votes=371/371  (runner-up 97)
S6: K4=0x3a  votes=388/388  (runner-up 79)
S7: K4=0x1f  votes=358/358  (runner-up 114)
S8: K4=0x23  votes=363/363  (runner-up 115)

Queries used: 801
Recovered last-round subkey K4 = 896E619BA7E3


## Step 4: reconstruct the full key and verify

`K4 = PC2(rotl(C0,6) || rotl(D0,6))`. PC-2 keeps 48 of the 56 key-schedule
bits, so 8 bits are unknown; we undo the rotations, brute-force the 8 missing
bits against one known plaintext/ciphertext pair, then map the result back
through PC-1 to the 14-hex key and verify on fresh plaintexts.

In [5]:
def rotr_list(bits, n):
    return bits[-n:] + bits[:-n]

TOTAL_SHIFT = sum(SHIFT_SCHEDULE)     # 6 for 4 rounds

def subkeys_from_combined0(c0):
    left, right = c0[:28], c0[28:]
    subs = []
    for shift in SHIFT_SCHEDULE:
        left = left_shift(left, shift)
        right = left_shift(right, shift)
        subs.append(permute(left + right, PC2))
    return subs

def encrypt_with_combined0(pt64, c0):
    subs = subkeys_from_combined0(c0)
    block = [str(b) for b in bits_of(pt64, 64)]
    left, right = block[:32], block[32:]
    for sk in subs:
        exp = permute(right, E)
        xr = [str(int(exp[i]) ^ int(sk[i])) for i in range(48)]
        sub = []
        for i in range(8):
            ch = xr[i * 6:(i + 1) * 6]
            r = int(ch[0] + ch[5], 2)
            c = int(''.join(ch[1:5]), 2)
            sub.extend(list(to_bin(S_BOXES[i][r][c], 4)))
        f_out = permute(sub, P)
        right, left = [str(int(left[i]) ^ int(f_out[i])) for i in range(32)], right
    return int(''.join(right + left), 2)

K4_bits = bits_of(K4_rec, 48)
combined4 = [None] * 56
for k in range(48):
    combined4[PC2[k] - 1] = K4_bits[k]
unknown_pos = [i for i in range(56) if combined4[i] is None]

kp_pt = random.getrandbits(64)
kp_ct = oracle(kp_pt)

combined0 = None
for guess in range(1 << len(unknown_pos)):
    c4 = list(combined4)
    for idx, pos in enumerate(unknown_pos):
        c4[pos] = (guess >> (len(unknown_pos) - 1 - idx)) & 1
    c0 = rotr_list(c4[:28], TOTAL_SHIFT) + rotr_list(c4[28:], TOTAL_SHIFT)
    c0 = [str(b) for b in c0]
    if encrypt_with_combined0(kp_pt, c0) == kp_ct:
        combined0 = c0
        break
assert combined0 is not None, 'no key candidate matched'

key64 = [0] * 64                       # invert PC-1 (parity bits set to 0)
for k in range(56):
    key64[PC1[k] - 1] = int(combined0[k])
key56 = [str(key64[i]) for i in range(64) if (i + 1) % 8 != 0]
recovered_key_hex = f"{int(''.join(key56), 2):014X}"

# Verify: encrypt fresh random plaintexts locally with the recovered key and
# compare against the live oracle. Agreement on independent inputs confirms the
# recovered key matches the oracle's hidden key.
ok = all(
    des_encrypt_hex(f'{(pt := random.getrandbits(64)):016X}', recovered_key_hex)
    == f'{oracle(pt):016X}'
    for _ in range(20)
)
print(f'RECOVERED ORACLE KEY (hex) = {recovered_key_hex}')
print(f'Verification on fresh plaintexts: {"PASS" if ok else "FAIL"}')
print(f'Total oracle queries used: {_query_count}')

RECOVERED ORACLE KEY (hex) = 8B2F88A663CB4D
Verification on fresh plaintexts: PASS
Total oracle queries used: 822
